In [1]:
import pandas as pd
import numpy as np
import datetime as dt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from dtw import dtw
import copy

from keras.models import Sequential
import matplotlib.pyplot as plt
%matplotlib inline

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector, TimeDistributed, Dropout

import plotly.express as px

ImportError: cannot import name 'runtime_version' from 'google.protobuf' (c:\Users\kashika.p\AppData\Local\miniconda3\envs\baseline_model\Lib\site-packages\google\protobuf\__init__.py)

In [2]:
machines= ['SL40309_015']
programs= ['O0005(5303-005-C)']
crs= ['CR1_to_CR4', 'CR5_to_CR8', 'CR9_to_CR12', 'CR12_to_CR16']
base_name = 'data'
paths = []

# data_SL40309_015_O0005(5303-005-C)_CR12_to_CR16

for machine in machines:
    for program in programs:
        for cr in crs:
            path = f'{base_name}_{machine}_{program}_{cr}.csv'
            paths.append(path)
print(paths)

dfs = []

for path in paths:
    data = pd.read_csv(f'../data/{path}')
    dfs.append(data)
    # print(len(data))
    
data = pd.concat(dfs, ignore_index=True)

data = data.pivot(index=['custom_id', 'timestamp', 'status', 'cr'], columns=['name', 'workstationcomponent'], values='value')
data = data.reset_index()
data.columns = [f'{col}_{comp}' if comp != '' else col for col, comp in data.columns]
data.columns

['data_SL40309_015_O0005(5303-005-C)_CR1_to_CR4.csv', 'data_SL40309_015_O0005(5303-005-C)_CR5_to_CR8.csv', 'data_SL40309_015_O0005(5303-005-C)_CR9_to_CR12.csv', 'data_SL40309_015_O0005(5303-005-C)_CR12_to_CR16.csv']


Index(['custom_id', 'timestamp', 'status', 'cr', 'gcode.ncode_Path_Path_1',
       'position_Linear_Z', 'toolnumber_Path_Path_1', 'position_Linear_X',
       'gcode.toolSlotId_Path_Path_1', 'programcomment_Path_Path_1',
       'pathfeedrate_Path_Path_1', 'gcode.program_Path_Path_1',
       'gcode.mcode_Path_Path_1', 'execution_Path_Path_1',
       'spindlespeed_actual_Rotary_C5', 'load_Rotary_C5',
       'spindlerotating_Rotary_C5', 'load_Linear_X',
       'gcode.gearselect_Path_Path_1', 'counter.last30seventscount_nan',
       'gcode.coolant_Path_Path_1', 'coolant_Coolant_Coolant', 'load_Linear_Z',
       'rapidoverride_Path_Path_1', 'controllermode_Path_Path_1',
       'operationmode_Path_Path_1', 'cutting_Path_Path_1', 'heartbeat_datatap',
       'line_Path_Path_1', 'partcount_Path_Path_1',
       'gcode.programname_Path_Path_1', 'fault_Path_Path_1',
       'gcode.spindle_Path_Path_1', 'jogoverride_Path_Path_1'],
      dtype='str')

In [3]:
split_columns = data['status'].str.split('/', expand=True)
data[['program_name', 'nsequence', 'execution']] = split_columns[[0,1,2]]

data['load_Rotary_C5'] = data['load_Rotary_C5'].astype(float)
data['position_Linear_Z'] = data['position_Linear_Z'].astype(float)
data['position_Linear_X'] = data['position_Linear_X'].astype(float)
data['spindlespeed_actual_Rotary_C5'] = data['spindlespeed_actual_Rotary_C5'].astype(float)
data['pathfeedrate_Path_Path_1'] = data['pathfeedrate_Path_Path_1'].astype(float)

data = data.ffill()

In [4]:
# datapoints where machining == True
data = data[data['timestamp'] <'2024-04-12']
data = data[(data['execution'] == 'ACTIVE') & 
            (data['spindlespeed_actual_Rotary_C5'] != 0) & 
            (data['pathfeedrate_Path_Path_1'] != 0) & 
            (data['load_Rotary_C5'] > 0) & 
            (data['pathfeedrate_Path_Path_1'] <= 1000)]

In [5]:
len(data)

85606

In [6]:
data.tail(5)

,custom_id,timestamp,status,cr,gcode.ncode_Path_Path_1,position_Linear_Z,toolnumber_Path_Path_1,position_Linear_X,gcode.toolSlotId_Path_Path_1,programcomment_Path_Path_1,...,heartbeat_datatap,line_Path_Path_1,partcount_Path_Path_1,gcode.programname_Path_Path_1,fault_Path_Path_1,gcode.spindle_Path_Path_1,jogoverride_Path_Path_1,program_name,nsequence,execution
119737,20097830,2024-03-12 06:39:53.838287,O0005(5303-005-C)/N40/ACTIVE,CR-16,N40G04X2000,14.1226,1212,19.1076,G0T1212,ROUGH OD ONLY,...,1709247815674,10,886,O0005(5303-005-C),OVER TRAVEL : -,stop,90,O0005(5303-005-C),N40,ACTIVE
119738,20097831,2024-03-12 06:39:53.838287,O0005(5303-005-C)/N40/ACTIVE,CR-16,N40G04X2000,14.1226,1212,-0.0885,G0T1212,ROUGH OD ONLY,...,1709247815674,10,886,O0005(5303-005-C),OVER TRAVEL : -,stop,90,O0005(5303-005-C),N40,ACTIVE
119739,20097832,2024-03-12 06:39:53.838287,O0005(5303-005-C)/N40/ACTIVE,CR-16,N40G04X2000,14.1226,1212,-0.0885,G0T1212,ROUGH OD ONLY,...,1709247815674,10,886,O0005(5303-005-C),OVER TRAVEL : -,stop,90,O0005(5303-005-C),N40,ACTIVE
119740,20097833,2024-03-12 06:39:53.838287,O0005(5303-005-C)/N40/ACTIVE,CR-16,N40G04X2000,-5.6411,1212,-0.0885,G0T1212,ROUGH OD ONLY,...,1709247815674,10,886,O0005(5303-005-C),OVER TRAVEL : -,stop,90,O0005(5303-005-C),N40,ACTIVE
119741,20097834,2024-03-12 06:39:53.838287,O0005(5303-005-C)/N40/ACTIVE,CR-16,N40G04X2000,-5.6411,1212,-0.0885,G0T1212,ROUGH OD ONLY,...,1709247815674,10,886,O0005(5303-005-C),OVER TRAVEL : -,stop,90,O0005(5303-005-C),N40,ACTIVE


In [7]:

dfN10_load = data[['timestamp', 'load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution', 'program_name', 'nsequence', 'cr']]

In [8]:
len(dfN10_load)

85606

In [9]:
dfN10_load = dfN10_load.drop_duplicates(subset=['timestamp'], keep ='first')
dfN10_load.reset_index(drop=True)
len(dfN10_load)

9937

In [10]:
dfN10_load['index'] = dfN10_load.groupby(['cr']).cumcount()

In [11]:
fig = px.line(dfN10_load, x="index", y="load_Rotary_C5", color='cr')
fig.update_layout(title_text="Load Vs Index")
fig.show()

In [12]:
LE = LabelEncoder()

dfN10_load['execution'] = LE.fit_transform(dfN10_load['execution'])
dfN10_load.fillna(0, inplace=True)

,timestamp,load_Rotary_C5,position_Linear_X,position_Linear_Z,spindlespeed_actual_Rotary_C5,pathfeedrate_Path_Path_1,execution,program_name,nsequence,cr,index
46,2024-03-11 14:01:19.408233,2.0,18.8794,12.5102,35.0,238.02,0,O0005(5303-005-C),N10,CR-1,0
49,2024-03-11 14:01:29.432814,2.0,18.8794,12.5102,31.0,238.02,0,O0005(5303-005-C),N10,CR-1,1
63,2024-03-11 14:01:37.451878,1.0,16.9492,4.7948,34.0,238.02,0,O0005(5303-005-C),N10,CR-1,2
64,2024-03-11 14:01:44.468672,1.0,16.9492,4.7948,33.0,238.02,0,O0005(5303-005-C),N10,CR-1,3
78,2024-03-11 14:01:52.489886,1.0,16.5496,3.1977,34.0,14.81,0,O0005(5303-005-C),N10,CR-1,4
...,...,...,...,...,...,...,...,...,...,...,...
119559,2024-03-12 06:39:29.791527,1.0,16.2506,-0.1245,35.0,0.49,0,O0005(5303-005-C),N40,CR-16,524
119565,2024-03-12 06:39:35.803604,1.0,16.2758,-0.1586,29.0,1.39,0,O0005(5303-005-C),N40,CR-16,525
119575,2024-03-12 06:39:45.824041,1.0,17.9761,1.7775,29.0,237.16,0,O0005(5303-005-C),N40,CR-16,526
119579,2024-03-12 06:39:49.830634,1.0,19.1076,7.9186,29.0,237.16,0,O0005(5303-005-C),N40,CR-16,527


In [13]:
scaler1 = MinMaxScaler()

dfN10_load[['load_Rotary_C5']]= scaler1.fit_transform(dfN10_load[['load_Rotary_C5']])

In [14]:
dfN10_load.head()

,timestamp,load_Rotary_C5,position_Linear_X,position_Linear_Z,spindlespeed_actual_Rotary_C5,pathfeedrate_Path_Path_1,execution,program_name,nsequence,cr,index
46,2024-03-11 14:01:19.408233,0.014493,18.8794,12.5102,35.0,238.02,0,O0005(5303-005-C),N10,CR-1,0
49,2024-03-11 14:01:29.432814,0.014493,18.8794,12.5102,31.0,238.02,0,O0005(5303-005-C),N10,CR-1,1
63,2024-03-11 14:01:37.451878,0.000000,16.9492,4.7948,34.0,238.02,0,O0005(5303-005-C),N10,CR-1,2
64,2024-03-11 14:01:44.468672,0.000000,16.9492,4.7948,33.0,238.02,0,O0005(5303-005-C),N10,CR-1,3
78,2024-03-11 14:01:52.489886,0.000000,16.5496,3.1977,34.0,14.81,0,O0005(5303-005-C),N10,CR-1,4


In [15]:
dfN10_load['cr'].value_counts()

cr
CR-1     783
CR-10    778
CR-13    744
CR-4     740
CR-2     665
CR-3     664
CR-11    655
CR-12    654
CR-8     538
CR-14    537
CR-5     535
CR-6     531
CR-7     531
CR-9     530
CR-16    529
CR-15    523
Name: count, dtype: int64

In [16]:
dfN10_load['nsequence'].value_counts()

nsequence
N20    3729
N30    3205
N10    2477
N40     526
Name: count, dtype: int64

In [17]:
def euclidean_distance(x, y):
    return abs(x - y)  

def dtw_cost_alignment(df):
    size = len(df['cr'].value_counts())
    dtw_costs = [[0 for _ in range(size)] for _ in range(size)]
    for i in range(size):
        for j in range(i, size):
            df_load_template = df.loc[df['cr'] == f'CR-{i+1}', 'load_Rotary_C5'].values
            df_load_template = df_load_template[~np.isnan(df_load_template)]

            df_load_query = df.loc[df['cr'] == f'CR-{j+1}', 'load_Rotary_C5'].values
            df_load_query = df_load_query[~np.isnan(df_load_query)]

            dtw_cost = dtw(df_load_template, df_load_query,dist=euclidean_distance)
            # print("test: ", dtw_cost[0])
            cost = round(dtw_cost[0],5)
            print(f'CR: {i+1}-{j+1} COST: {cost}')
            dtw_costs[i][j] = cost
            dtw_costs[j][i] = cost
    return dtw_costs


In [18]:
def cost_per_time_stamp(df, cost):
    costs_size = len(df['cr'].value_counts())
    new_costs = [[0 for _ in range(costs_size)] for _ in range(costs_size)]
    for i in range(costs_size):
        for j in range(i, costs_size):
            df_load_query = df.loc[df['cr'] == f'CR-{j+1}', 'load_Rotary_C5'].values
            df_load_query = df_load_query[~np.isnan(df_load_query)]

            size = len(df_load_query)
            new_cost = round(cost[i][j]/size, 5)
            
            new_costs[i][j] = new_cost
            new_costs[j][i] = new_cost
            
    return new_costs
    

In [19]:
n10_dtw_cost = dtw_cost_alignment(dfN10_load)

CR: 1-1 COST: 0.0
CR: 1-2 COST: 35.98551
CR: 1-3 COST: 41.07246
CR: 1-4 COST: 37.43478
CR: 1-5 COST: 32.2029
CR: 1-6 COST: 40.2029
CR: 1-7 COST: 28.97101
CR: 1-8 COST: 38.27536
CR: 1-9 COST: 34.75362
CR: 1-10 COST: 1.05797
CR: 1-11 COST: 33.6087
CR: 1-12 COST: 40.71014
CR: 1-13 COST: 37.42029
CR: 1-14 COST: 32.78261
CR: 1-15 COST: 38.91304
CR: 1-16 COST: 28.72464
CR: 2-2 COST: 0.0
CR: 2-3 COST: 15.10145
CR: 2-4 COST: 19.26087
CR: 2-5 COST: 13.81159
CR: 2-6 COST: 14.7971
CR: 2-7 COST: 13.88406
CR: 2-8 COST: 13.26087
CR: 2-9 COST: 17.04348
CR: 2-10 COST: 36.88406
CR: 2-11 COST: 2.65217
CR: 2-12 COST: 14.6087
CR: 2-13 COST: 18.86957
CR: 2-14 COST: 14.5942
CR: 2-15 COST: 14.2029
CR: 2-16 COST: 14.2029
CR: 3-3 COST: 0.0
CR: 3-4 COST: 19.44928
CR: 3-5 COST: 17.04348
CR: 3-6 COST: 11.71014
CR: 3-7 COST: 18.15942
CR: 3-8 COST: 13.5942
CR: 3-9 COST: 16.31884
CR: 3-10 COST: 41.52174
CR: 3-11 COST: 15.69565
CR: 3-12 COST: 0.46377
CR: 3-13 COST: 18.85507
CR: 3-14 COST: 18.14493
CR: 3-15 COST: 11.7

In [20]:
print(n10_dtw_cost)

[[np.float64(0.0), np.float64(35.98551), np.float64(41.07246), np.float64(37.43478), np.float64(32.2029), np.float64(40.2029), np.float64(28.97101), np.float64(38.27536), np.float64(34.75362), np.float64(1.05797), np.float64(33.6087), np.float64(40.71014), np.float64(37.42029), np.float64(32.78261), np.float64(38.91304), np.float64(28.72464)], [np.float64(35.98551), np.float64(0.0), np.float64(15.10145), np.float64(19.26087), np.float64(13.81159), np.float64(14.7971), np.float64(13.88406), np.float64(13.26087), np.float64(17.04348), np.float64(36.88406), np.float64(2.65217), np.float64(14.6087), np.float64(18.86957), np.float64(14.5942), np.float64(14.2029), np.float64(14.2029)], [np.float64(41.07246), np.float64(15.10145), np.float64(0.0), np.float64(19.44928), np.float64(17.04348), np.float64(11.71014), np.float64(18.15942), np.float64(13.5942), np.float64(16.31884), np.float64(41.52174), np.float64(15.69565), np.float64(0.46377), np.float64(18.85507), np.float64(18.14493), np.float6

In [21]:
n10_dtw_cost = [[0.0, 35.98551, 41.07246, 37.43478, 32.2029, 40.2029, 28.97101, 38.27536, 34.75362, 1.05797, 33.6087, 40.71014, 37.42029, 32.78261, 38.91304, 28.72464], [35.98551, 0.0, 15.10145, 19.26087, 13.81159, 14.7971, 13.88406, 13.26087, 17.04348, 36.88406, 2.65217, 14.6087, 18.86957, 14.5942, 14.2029, 14.2029], [41.07246, 15.10145, 0.0, 19.44928, 17.04348, 11.71014, 18.15942, 13.5942, 16.31884, 41.52174, 15.69565, 0.46377, 18.85507, 18.14493, 11.76812, 17.95652], [37.43478, 19.26087, 19.44928, 0.0, 12.23188, 13.43478, 16.46377, 15.52174, 12.04348, 38.43478, 16.95652, 18.7971, 0.78261, 12.81159, 12.5942, 16.18841], [32.2029, 13.81159, 17.04348, 12.23188, 0.0, 10.82609, 12.13043, 12.49275, 9.44928, 33.15942, 12.98551, 16.3913, 12.86957, 1.42029, 10.13043, 11.86957], [40.2029, 14.7971, 11.71014, 13.43478, 10.82609, 0.0, 12.65217, 11.07246, 9.23188, 39.71014, 14.30435, 11.24638, 12.72464, 12.24638, 0.82609, 12.50725], [28.97101, 13.88406, 18.15942, 16.46377, 12.13043, 12.65217, 0.0, 10.7971, 12.62319, 29.81159, 12.46377, 17.72464, 16.55072, 12.71014, 12.43478, 0.31884], [38.27536, 13.26087, 13.5942, 15.52174, 12.49275, 11.07246, 10.7971, 0.0, 10.81159, 39.11594, 13.23188, 13.0, 15.31884, 12.43478, 10.71014, 10.73913], [34.75362, 17.04348, 16.31884, 12.04348, 9.44928, 9.23188, 12.62319, 10.81159, 0.0, 35.66667, 14.50725, 15.7971, 12.55072, 10.02899, 8.46377, 12.18841], [1.05797, 36.88406, 41.52174, 38.43478, 33.15942, 39.71014, 29.81159, 39.11594, 35.66667, 0.0, 34.50725, 41.17391, 38.42029, 33.6087, 38.88406, 29.56522], [33.6087, 2.65217, 15.69565, 16.95652, 12.98551, 14.30435, 12.46377, 13.23188, 14.50725, 34.50725, 0.0, 15.2029, 16.71014, 13.56522, 13.71014, 12.13043], [40.71014, 14.6087, 0.46377, 18.7971, 16.3913, 11.24638, 17.72464, 13.0, 15.7971, 41.17391, 15.2029, 0.0, 18.2029, 17.49275, 11.24638, 17.4058], [37.42029, 18.86957, 18.85507, 0.78261, 12.86957, 12.72464, 16.55072, 15.31884, 12.55072, 38.42029, 16.71014, 18.2029, 0.0, 13.44928, 12.86957, 16.26087], [32.78261, 14.5942, 18.14493, 12.81159, 1.42029, 12.24638, 12.71014, 12.43478, 10.02899, 33.6087, 13.56522, 17.49275, 13.44928, 0.0, 11.55072, 12.44928], [38.91304, 14.2029, 11.76812, 12.5942, 10.13043, 0.82609, 12.43478, 10.71014, 8.46377, 38.88406, 13.71014, 11.24638, 12.86957, 11.55072, 0.0, 12.11594], [28.72464, 14.2029, 17.95652, 16.18841, 11.86957, 12.50725, 0.31884, 10.73913, 12.18841, 29.56522, 12.13043, 17.4058, 16.26087, 12.44928, 12.11594, 0.0]]

In [22]:
def get_crs_within_threshold(cost, threshold):
    size = len(cost)
    arr = copy.deepcopy(cost)
    crs_in_threshold = [[set([i + 1, j + 1]) for j in range(size)] for i in range(size)]
#     print(crs_in_threshold)
    max_index_i = 0
    max_index_j = 0
    for gap in range(2, size):
        for i, j in zip(range(size - gap), range(gap, size)): 
            cr_i = i + 1,
            cr_j = j + 1,
            min_cost = min(arr[i][j], arr[i][j - 1], arr[i + 1][j])
            if arr[i][j] <= threshold:
                if arr[i][j - 1] <= threshold and arr[i + 1][j] <= threshold:
#                     print(i, j, crs_in_threshold[i][j], crs_in_threshold[i][j - 1], crs_in_threshold[i + 1][j])
                    crs_in_threshold[i][j] = crs_in_threshold[i][j].union(crs_in_threshold[i][j - 1])
                    crs_in_threshold[i][j] = crs_in_threshold[i][j].union(crs_in_threshold[i + 1][j])
            else:
                min_cost = min(arr[i][j - 1], arr[i + 1][j])
                if arr[i][j - 1] == min_cost and arr[i][j - 1] <= threshold:
#                     arr[i][j] = min_cost
                    crs_in_threshold[i][j] = crs_in_threshold[i][j - 1]
                elif arr[i + 1][j] == min_cost and arr[i + 1][j] <= threshold:
#                     arr[i][j] = min_cost
                    crs_in_threshold[i][j] = crs_in_threshold[i + 1 ][j]
                else:
                    crs_in_threshold[i][j] = set([])
            if len(crs_in_threshold[i][j]) > len(crs_in_threshold[max_index_i][max_index_j]):
                max_index_i = i
                max_index_j = j
    return crs_in_threshold[max_index_i][max_index_j]  
    

In [23]:
n10_dtw_cost_per_timestamp = cost_per_time_stamp(dfN10_load, n10_dtw_cost)

In [24]:
def display_crs(costs, df, threshold):
    cr_set = get_crs_within_threshold(costs, threshold)
    print('num CRs: ', len(cr_set))
    
    df['cr_numeric'] = df['cr'].str.extract(r'(\d+)')
    df['cr_numeric'] = pd.to_numeric(df['cr_numeric'])
    filtered_df = df[df['cr_numeric'].isin(cr_set)]
    filtered_df = filtered_df.reset_index(drop=True)
    # filtered_df['index'] = filtered_df.groupby(['nsequence', 'cr_numeric']).cumcount()
    filtered_df['index'] = filtered_df.groupby(['cr_numeric']).cumcount()
#     print(filtered_df['cr'].value_counts())
    
    fig = px.line(filtered_df, x="index", y="load_Rotary_C5", color='cr')
    fig.update_layout(title_text="Load Vs Index")
    fig.show()
    return cr_set, filtered_df

In [25]:
# Nsequence: N10
# Threshold: 0.003
df10_cr_set, dfN10_load_filtered_cr = display_crs(n10_dtw_cost_per_timestamp, dfN10_load, 0.03)

num CRs:  11


In [26]:
df10_cr_set = list(df10_cr_set)
size = len(df10_cr_set)

train_ratio = 0.8
train_size = int(size*train_ratio)
test_size = size - train_size

print('train_size: ', train_size)
print('test_size: ', test_size)

train_crs = df10_cr_set[0: train_size]
test_crs = df10_cr_set[train_size: -1]

print('train_crs: ', train_crs)
print('test_crs: ', test_crs)

train_size:  8
test_size:  3
train_crs:  [2, 3, 4, 5, 6, 7, 8, 9]
test_crs:  [11, 12]


In [27]:
def to_sequences(x, seq_size):
    x_values = []
    y_values = []
    for i in range(len(x) - seq_size):
        x_values.append(x[i : (i + seq_size)])
        y_values.append(x[i + seq_size])
    return np.array(x_values), np.array(y_values)

In [28]:
dfN10_train_data = dfN10_load_filtered_cr[dfN10_load_filtered_cr['cr_numeric'].isin(train_crs)]
len(dfN10_train_data)

4734

In [29]:
dfN10_test_data = dfN10_load_filtered_cr[dfN10_load_filtered_cr['cr_numeric'].isin(test_crs)]
len(dfN10_test_data)

1309

In [30]:
dfN10_train_data = dfN10_train_data[['load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution',]]
dfN10_test_data = dfN10_test_data[['load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution',]]

In [31]:
scaler2 = MinMaxScaler()
data_scaled = scaler2.fit(dfN10_train_data)
# Scaling dataset according to weights of train data
dfN10_train_data_scaled = data_scaled.transform(dfN10_train_data)
dfN10_test_data_scaled = data_scaled.transform(dfN10_test_data)

In [32]:
df10_trainX, df10_trainY = to_sequences(dfN10_train_data_scaled, 10)
print(df10_trainX.shape, df10_trainY.shape)

df10_testX, df10_testY = to_sequences(dfN10_test_data_scaled, 10)
print(df10_testX.shape, df10_testY.shape)

(4724, 10, 6) (4724, 6)
(1299, 10, 6) (1299, 6)


In [33]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, RepeatVector, Dense, TimeDistributed
from tensorflow.keras.models import Model

class LSTMTimeSeriesAutoencoder:
    def __init__(self, sequence_length, num_dimensions, dim_weights):
        self.sequence_length = sequence_length
        self.num_dimensions = num_dimensions
        self.dim_weights = dim_weights
        self.model = self.build_model()

    def build_model(self):
        # Define input shape
        input_shape = (self.sequence_length, self.num_dimensions)

        # Encoder
        encoder_inputs = Input(shape=input_shape)
        encoder = LSTM(128, return_sequences=True)(encoder_inputs)
        encoder = LSTM(64, return_sequences=True)(encoder)
        encoder = LSTM(32, return_sequences=True)(encoder)
        encoder = LSTM(16, return_sequences=True)(encoder)
        encoder = LSTM(8, return_sequences=False)(encoder)

        # Repeat the latent vector for each time step
        decoder_inputs = RepeatVector(self.sequence_length)(encoder)

        # Decoder
        decoder = LSTM(8, return_sequences=True)(decoder_inputs)
        decoder = LSTM(16, return_sequences=True)(decoder)
        decoder = LSTM(32, return_sequences=True)(decoder)
        decoder = LSTM(64, return_sequences=True)(decoder)
        decoder = LSTM(128, return_sequences=True)(decoder)

        # Output layer
        output = TimeDistributed(Dense(self.num_dimensions))(decoder)

        # Define the model
        model = Model(inputs=encoder_inputs, outputs=output)

        return model

    def custom_loss(self, y_true, y_pred):
        # Define weights for each dimension
        weights = tf.constant(self.dim_weights, dtype=tf.float32)

        # Calculate mean squared error (MSE) for each dimension
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=0)

        # Multiply MSE by weights and sum across dimensions
        weighted_loss = tf.reduce_sum(mse * weights)

        return weighted_loss

    def compile_model(self):
        self.model.compile(optimizer='adam', loss=self.custom_loss)

    def summary(self):
        self.model.summary()

    def train(self, x_train, epochs, batch_size, callbacks):
        history = self.model.fit(x_train, x_train, epochs=epochs, batch_size=batch_size, verbose=1, validation_split=0.1,  callbacks=callbacks, shuffle=False)
        return history

    def predict(self, x):
        return self.model.predict(x)

    def save_weights(self, filepath):
        # Save model weights
        self.model.save_weights(filepath)

    def load_weights(self, filepath):
        # Load model weights
        self.model.load_weights(filepath)


In [34]:
from keras.callbacks import ModelCheckpoint

# Define the ModelCheckpoint callback
checkpoint = ModelCheckpoint(filepath='models/LSTM_Autoencoder_30_April_2024_program_level.keras', 
                             monitor='val_loss', 
                             verbose=1, 
                             save_best_only=True, 
                             mode='min')

In [35]:
model = LSTMTimeSeriesAutoencoder(sequence_length= df10_trainX.shape[1], num_dimensions=df10_trainX.shape[2], dim_weights = [0.8, 0.05, 0.05, 0.05, 0.025, 0.025])

In [36]:
model.compile_model()
history = model.train(x_train=df10_trainX, epochs=50, batch_size=64, callbacks=[checkpoint])

Epoch 1/50
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 0.7101
Epoch 1: val_loss improved from None to 0.47726, saving model to models/LSTM_Autoencoder_30_April_2024_program_level.keras

Epoch 1: finished saving model to models/LSTM_Autoencoder_30_April_2024_program_level.keras
67/67 ━━━━━━━━━━━━━━━━━━━━ 15s 68ms/step - loss: 0.6194 - val_loss: 0.4773
Epoch 2/50
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.3790
Epoch 2: val_loss improved from 0.47726 to 0.28235, saving model to models/LSTM_Autoencoder_30_April_2024_program_level.keras

Epoch 2: finished saving model to models/LSTM_Autoencoder_30_April_2024_program_level.keras
67/67 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 0.3509 - val_loss: 0.2823
Epoch 3/50
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.2711
Epoch 3: val_loss improved from 0.28235 to 0.18094, saving model to models/LSTM_Autoencoder_30_April_2024_program_level.keras

Epoch 3: finished saving model to models/LSTM_Autoencoder_30_April_2024_program_level.keras

In [37]:
model = LSTMTimeSeriesAutoencoder(sequence_length= df10_trainX.shape[1], num_dimensions=df10_trainX.shape[2], dim_weights = [0.8, 0.05, 0.05, 0.05, 0.025, 0.025])
# Load model weights
model.load_weights('models/LSTM_Autoencoder_30_April_2024_program_level.keras')

In [38]:
predicted_load_train = model.predict(df10_trainX)

148/148 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step


In [39]:
df10_trainX[10:20,4,0]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [40]:
predicted_load_train[10:20, 4, 0]

array([-0.00123411, -0.00127915, -0.00138026, -0.00139014, -0.00138451,
       -0.00136684, -0.00133717, -0.00255347, -0.00251155, -0.00277312],
      dtype=float32)

In [41]:
def get_mean_and_std(x,y,t):
    error = x[:,t,0] - y[:, t, 0]
    mean_rmse = np.sqrt(np.mean(np.square(error)))
    std_rmse = np.std(error)
    return mean_rmse, std_rmse
    

In [42]:
def mark_anomaly(x, y, t, mean, std, relax=1):
    error = x[:,t,0] - y[:, t, 0]
    deviation = np.abs(error)

    threshold = mean + relax*std
    anomalies = deviation > threshold

    # print(mean_rmse, std_rmse, error, deviation)
    # print(anomalies)
    return anomalies

In [43]:
mean, std =  get_mean_and_std(df10_trainX, predicted_load_train, 4)
print(mean, std)

0.0526633285618584 0.05266297953101322


In [44]:
anomaly = mark_anomaly(df10_trainX, predicted_load_train, 4, mean, std)

In [45]:
def display_prediction(df10_trainX, predicted_load_train, t, mean, std, relax):
    
    anomaly=mark_anomaly(df10_trainX, predicted_load_train, t, mean, std, relax)
    # print()
    # Create DataFrames for actual and predicted values
    # print(anomaly)
    df1 = pd.DataFrame({'x': range(len(df10_trainX)), 'y': df10_trainX[:, t, 0], 'color': 'Actual', 'anomaly': anomaly})
    df2 = pd.DataFrame({'x': range(len(predicted_load_train)), 'y': predicted_load_train[:, t, 0], 'color': 'Predicted', 'anomaly': False})
    # df3 = pd.DataFrame({'x': range(len(predicted_load_train)), 'y': predicted_load_train[:, t, 0], 'color': 'Anomaly', 'anomaly':anomaly})
    
    # Concatenate the DataFrames
    df = pd.concat([df1, df2])
    # print(df)
    # Plot both actual and predicted values
    fig = px.line(df, x='x', y='y', color='color')

    df_anomaly = df[df['anomaly']==True]
    print("Number of anomalies: ", len(df_anomaly))
    scatter_data = px.scatter(df_anomaly, x='x', y='y', color_discrete_sequence=['black']).data[0]
    scatter_data.update(marker=dict(size=5))
    fig.add_trace(scatter_data)

    fig.show()


In [46]:
display_prediction(df10_trainX, predicted_load_train, 4, mean, std, 2)

Number of anomalies:  84


In [47]:
predicted_load_test = model.predict(df10_testX)

41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


In [48]:
# results on test data 

In [49]:
display_prediction(df10_testX, predicted_load_test, 4, mean, std, 2)

Number of anomalies:  23


In [50]:
def duplicate_rows_with_probability(arr, probability=0.1, duplicate_count=50):
    duplicated_rows = []
    for row in arr:
        t = np.random.rand()
        if t < probability:
            for i in range(duplicate_count):
                duplicated_rows.append(row)
            # duplicated_rows.append(row)
        elif t >= probability and t < min(0.5, 2*probability):
            pass
        else:
            duplicated_rows.append(row)
        duplicated_rows.append(row)
            
    duplicated_arr = np.array(duplicated_rows)
    return duplicated_arr

In [51]:
def graph_for_dilation(dfN10_train_data_scaled_2, prob, duplicate_count, mean, std, relax):
    dilated_test = duplicate_rows_with_probability(dfN10_train_data_scaled_2, prob, duplicate_count)
    # print(dilated_test.shape)
    # df10_trainX_3, _ = to_sequences(dilated_test, 10)
    # print(df10_trainX_3.shape)
    predicted_load_train_3 = model.predict(dilated_test)
    anomalies=mark_anomaly(dilated_test, predicted_load_train_3, 4, mean, std, relax)
    display_prediction(dilated_test, predicted_load_train_3, 4, mean, std, relax)

In [52]:
graph_for_dilation(df10_testX, 0.3, 2, mean, std, 2)

86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Number of anomalies:  44


In [53]:
# impact with time dilation

In [54]:
graph_for_dilation(df10_testX, 0, 0, mean, std, 2)

82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
Number of anomalies:  46


In [55]:
def get_noise(x, noise_level=0.1):
    size = len(x)
    noise = np.random.normal(loc=0, scale=noise_level, size=x.shape)
    return noise
    

In [56]:
noise = get_noise(df10_testX, 0.00)

In [57]:
predicted_load_test_noise = model.predict(df10_testX + noise)

41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step


In [58]:
display_prediction(df10_testX + noise, predicted_load_test_noise, 4,  mean, std, 2)

Number of anomalies:  23


In [59]:
noise = get_noise(df10_testX, 0.01)
predicted_load_test_noise = model.predict(df10_testX + noise)
display_prediction(df10_testX + noise, predicted_load_test_noise, 4,  mean, std, 2)

41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
Number of anomalies:  24


In [60]:
noise = get_noise(df10_testX, 0.02)
predicted_load_test_noise = model.predict(df10_testX + noise)
display_prediction(df10_testX + noise, predicted_load_test_noise, 4,  mean, std, 2)

41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
Number of anomalies:  26


In [61]:
noise = get_noise(df10_testX, 0.03)
predicted_load_test_noise = model.predict(df10_testX + noise)
display_prediction(df10_testX + noise, predicted_load_test_noise, 4,  mean, std, 2)

41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
Number of anomalies:  31


In [62]:
# impact with synthetic noise

In [63]:
noise = get_noise(df10_testX, 0.1)
predicted_load_test_noise = model.predict(df10_testX + noise)
display_prediction(df10_testX + noise, predicted_load_test_noise, 4,  mean, std, 2)

41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
Number of anomalies:  149


In [64]:
def get_patch_noise(x, random_patches=None, reduce_intensity=1):
    size = len(x)
    noise = np.zeros_like(x)
    # noise = np.ones(x.shape)/reduce_intensity
    if random_patches:
        for start, end in random_patches:
            patch_size = end - start
            print([*x.shape[-2:]] + [patch_size])
            # patch_noise = np.random.random(*x.shape[-2:], patch_size)
            patch_noise = np.random.random([*x.shape[-2:]] + [patch_size])/reduce_intensity
            # patch_noise = np.ones([*x.shape[-2:]] + [patch_size])/reduce_intensity
            # print(patch_noise)
            noise[start:end] = np.transpose(patch_noise, axes=(2, 0, 1)) 
    return noise


In [65]:
def get_ones_noise(x, random_patches=None, reduce_intensity=1):
    size = len(x)
    noise = np.zeros_like(x)
    noise = np.ones(x.shape)/reduce_intensity
    # if random_patches:
    #     for start, end in random_patches:
    #         patch_size = end - start
    #         print([*x.shape[-2:]] + [patch_size])
    #         # patch_noise = np.random.random(*x.shape[-2:], patch_size)
    #         patch_noise = np.random.random([*x.shape[-2:]] + [patch_size])/reduce_intensity
    #         # patch_noise = np.ones([*x.shape[-2:]] + [patch_size])/reduce_intensity
    #         # print(patch_noise)
    #         noise[start:end] = np.transpose(patch_noise, axes=(2, 0, 1))  
    
    return noise


In [66]:
# impact with inflated load points 

In [67]:
noise = get_ones_noise(df10_testX, [(50, 70), (300, 350)], 5)
# print(noise)
predicted_load_test_noise = model.predict(df10_testX + noise)
display_prediction(df10_testX + noise, predicted_load_test_noise, 4,  mean, std, 2)

41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step
Number of anomalies:  128


In [68]:
# impact with patch noise

In [69]:
noise = get_patch_noise(df10_testX, [(70, 100), (400, 450), (600, 670), (800, 900)], 2)
noise_data = df10_testX + noise
predicted_load_test_noise = model.predict(noise_data)
display_prediction(noise_data, predicted_load_test_noise, 4,  mean, std, 2)

[10, 6, 30]
[10, 6, 50]
[10, 6, 70]
[10, 6, 100]
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Number of anomalies:  134


In [70]:
noise = get_patch_noise(df10_testX, [(70, 100), (400, 450), (600, 670), (800, 900)], 2)
# print(noise)
noise_data = df10_testX - noise
predicted_load_test_noise = model.predict(noise_data)
display_prediction(noise_data, predicted_load_test_noise, 4,  mean, std, 2)

[10, 6, 30]
[10, 6, 50]
[10, 6, 70]
[10, 6, 100]
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
Number of anomalies:  161


In [71]:
dfN10_train_data_rejected = dfN10_load[dfN10_load['cr'].isin(['CR-1','CR-10','CR-13','CR-14','CR-15'])]
len(dfN10_train_data_rejected)

3365

In [72]:
dfN10_train_data_rejected.cr_numeric.value_counts()

cr_numeric
1     783
10    778
13    744
14    537
15    523
Name: count, dtype: int64

In [73]:
dfN10_train_data_rejected.head()

,timestamp,load_Rotary_C5,position_Linear_X,position_Linear_Z,spindlespeed_actual_Rotary_C5,pathfeedrate_Path_Path_1,execution,program_name,nsequence,cr,index,cr_numeric
46,2024-03-11 14:01:19.408233,0.014493,18.8794,12.5102,35.0,238.02,0,O0005(5303-005-C),N10,CR-1,0,1
49,2024-03-11 14:01:29.432814,0.014493,18.8794,12.5102,31.0,238.02,0,O0005(5303-005-C),N10,CR-1,1,1
63,2024-03-11 14:01:37.451878,0.000000,16.9492,4.7948,34.0,238.02,0,O0005(5303-005-C),N10,CR-1,2,1
64,2024-03-11 14:01:44.468672,0.000000,16.9492,4.7948,33.0,238.02,0,O0005(5303-005-C),N10,CR-1,3,1
78,2024-03-11 14:01:52.489886,0.000000,16.5496,3.1977,34.0,14.81,0,O0005(5303-005-C),N10,CR-1,4,1


In [74]:
dfN10_train_data_rejected = dfN10_train_data_rejected.drop_duplicates(subset=['timestamp'], keep ='first')
dfN10_train_data_rejected.reset_index(drop=True)
len(dfN10_train_data_rejected)

3365

In [75]:
LE = LabelEncoder()

dfN10_train_data_rejected['execution'] = LE.fit_transform(dfN10_train_data_rejected['execution'])
dfN10_train_data_rejected.fillna(0, inplace=True)

,timestamp,load_Rotary_C5,position_Linear_X,position_Linear_Z,spindlespeed_actual_Rotary_C5,pathfeedrate_Path_Path_1,execution,program_name,nsequence,cr,index,cr_numeric
46,2024-03-11 14:01:19.408233,0.014493,18.8794,12.5102,35.0,238.02,0,O0005(5303-005-C),N10,CR-1,0,1
49,2024-03-11 14:01:29.432814,0.014493,18.8794,12.5102,31.0,238.02,0,O0005(5303-005-C),N10,CR-1,1,1
63,2024-03-11 14:01:37.451878,0.000000,16.9492,4.7948,34.0,238.02,0,O0005(5303-005-C),N10,CR-1,2,1
64,2024-03-11 14:01:44.468672,0.000000,16.9492,4.7948,33.0,238.02,0,O0005(5303-005-C),N10,CR-1,3,1
78,2024-03-11 14:01:52.489886,0.000000,16.5496,3.1977,34.0,14.81,0,O0005(5303-005-C),N10,CR-1,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...
113970,2024-03-12 05:52:55.273441,0.000000,16.2762,-0.1589,29.0,1.39,0,O0005(5303-005-C),N40,CR-15,518,15
113979,2024-03-12 05:53:04.289594,0.000000,17.9876,1.8090,29.0,237.18,0,O0005(5303-005-C),N40,CR-15,519,15
113983,2024-03-12 05:53:08.296589,0.000000,19.1076,7.9816,29.0,237.18,0,O0005(5303-005-C),N40,CR-15,520,15
113988,2024-03-12 05:53:13.306267,0.000000,19.1076,14.1226,29.0,237.16,0,O0005(5303-005-C),N40,CR-15,521,15


In [76]:
# dfN20_train_data = dfN20_load[dfN20_load['cr'].isin(['CR-1','CR-2','CR-3','CR-4','CR-5'])]
# len(dfN20_train_data)

In [77]:
dfN10_train_data_rejected = dfN10_train_data_rejected[['load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution',]]


In [78]:

dfN10_train_data_rejected_scaled = data_scaled.transform(dfN10_train_data_rejected)

print(dfN10_train_data_rejected_scaled.shape)

(3365, 6)


In [79]:
dfN10_train_data_rejected_scaled

array([[0.01960784, 0.94727721, 0.74390879, 0.05263158, 0.98762304,
        0.        ],
       [0.01960784, 0.94727721, 0.74390879, 0.0430622 , 0.98762304,
        0.        ],
       [0.        , 0.85088096, 0.42770257, 0.05023923, 0.98762304,
        0.        ],
       ...,
       [0.        , 0.95867376, 0.55830967, 0.03827751, 0.98413424,
        0.        ],
       [0.        , 0.95867376, 0.80999102, 0.03827751, 0.98405117,
        0.        ],
       [0.74509804, 0.        , 0.        , 0.1507177 , 0.        ,
        0.        ]], shape=(3365, 6))

In [80]:
dfN10_train_data_rejected_scaled.max()

np.float64(1.352941176470588)

In [81]:
dfN10_train_data_scaled.max()

np.float64(1.0)

In [82]:
# CRS rejected by DTW 

In [83]:
df20_trainX, df20_trainY = to_sequences(dfN10_train_data_rejected_scaled, 10)
print(df20_trainX.shape, df20_trainY.shape)

(3355, 10, 6) (3355, 6)


In [84]:
predicted_load_train_20 = model.predict(df20_trainX)

105/105 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step


In [85]:
display_prediction(df20_trainX, predicted_load_train_20, 4, mean, std, 2)

Number of anomalies:  29


In [86]:
df_risk_comparision = dfN10_load[dfN10_load['cr'].isin(['CR-3','CR-15'])]
len(df_risk_comparision)

1187

In [87]:
df_risk_comparision = df_risk_comparision.drop_duplicates(subset=['timestamp'], keep ='first')
df_risk_comparision.reset_index(drop=True)
len(df_risk_comparision)

1187

In [88]:
df_risk_base = df_risk_comparision[df_risk_comparision['cr'].isin(['CR-3'])]
df_risk_test = df_risk_comparision[df_risk_comparision['cr'].isin(['CR-15'])]

In [89]:
df_risk_base = df_risk_base[['load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution',]]
df_risk_test = df_risk_test[['load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution',]]


In [90]:

df_risk_base_scaled = data_scaled.transform(df_risk_base)
df_risk_test_scaled = data_scaled.transform(df_risk_test)

print(df_risk_base_scaled.shape, df_risk_test_scaled.shape)

(664, 6) (523, 6)


In [91]:
df_risk_base_scaled.max()

np.float64(1.0)

In [92]:
df_risk_test_scaled.max()

np.float64(0.9929808530963159)

In [93]:
df_risk_base_scaled_x, df_risk_base_scaled_y = to_sequences(df_risk_base_scaled, 10)
print(df_risk_base_scaled_x.shape, df_risk_base_scaled_y.shape)

(654, 10, 6) (654, 6)


In [94]:
df_risk_test_scaled_x, df_risk_test_scaled_y = to_sequences(df_risk_test_scaled, 10)
print(df_risk_test_scaled_x.shape, df_risk_test_scaled_y.shape)

(513, 10, 6) (513, 6)


In [95]:
def mark_anomalies_current_method_on_production(x, y, t):
    anomalies = []
    count = 0
    for i in range(len(y)):
        if i >= len(x):
            anomalies.append(False)
            # break
        elif y[i, t, 0] > 1.25 * x[i, t, 0] or y[i, t, 0] < 0.75 * x[i, t, 0]:
            anomalies.append(True)
            count += 1
        else:
            anomalies.append(False)
    return np.array(anomalies)


In [96]:
anomalies = mark_anomalies_current_method_on_production(df_risk_base_scaled_x, df_risk_test_scaled_x, 4)
len(anomalies)

513

In [97]:
def display_seq(x, y, t):

    anomalies = mark_anomalies_current_method_on_production(x, y, t)
    df1 = pd.DataFrame({'x': range(len(x)), 'y': x[:, t, 0], 'color': 'Baseline', 'anomaly': False})
    df2 = pd.DataFrame({'x': range(len(y)), 'y': y[:, t, 0], 'color': 'New CR', 'anomaly': anomalies})

    df = pd.concat([df1, df2])
    fig = px.line(df, x='x', y='y', color='color')

    df_anomaly = df[df['anomaly']==True]
    print("Number of anomalies: ", len(df_anomaly))
    scatter_data = px.scatter(df_anomaly, x='x', y='y', color_discrete_sequence=['black']).data[0]
    scatter_data.update(marker=dict(size=5))
    fig.add_trace(scatter_data)

    fig.show()


In [98]:
def display_prediction_new(df10_trainX, predicted_load_train, baseline, t, mean, std, relax):
    
    anomaly=mark_anomaly(df10_trainX, predicted_load_train, t, mean, std, relax)
    # print()
    # Create DataFrames for actual and predicted values
    # print(anomaly)
    df1 = pd.DataFrame({'x': range(len(df10_trainX)), 'y': df10_trainX[:, t, 0], 'color': 'Actual Load', 'anomaly': anomaly})
    df2 = pd.DataFrame({'x': range(len(predicted_load_train)), 'y': predicted_load_train[:, t, 0], 'color': 'Predicted Baseline', 'anomaly': False})
    df3 = pd.DataFrame({'x': range(len(baseline)), 'y': baseline[:, t, 0], 'color': 'Old Baseline', 'anomaly':False})
    
    # Concatenate the DataFrames
    df = pd.concat([df1, df2, df3])
    # print(df)
    # Plot both actual and predicted values
    fig = px.line(df, x='x', y='y', color='color')

    df_anomaly = df[df['anomaly']==True]
    print("Number of anomalies: ", len(df_anomaly))
    scatter_data = px.scatter(df_anomaly, x='x', y='y', color_discrete_sequence=['black']).data[0]
    scatter_data.update(marker=dict(size=5))
    fig.add_trace(scatter_data)

    fig.show()


In [99]:
def mark_risk_new(current_load, new_base, t, mean, std, relax):

    tool_disengagement_counter = 0
    tool_overload_counter = 0
    roc_counter = 0

    num_tool_disengagement = 0
    num_tool_overload = 0
    num_roc = 0

    prev_base_load = None
    prev_current_load = None
    for index, (x, y) in enumerate(zip(current_load[:,t,0], new_base[:, t, 0])):
        error = x - y
        # print(error)
        if error > mean + relax*std:
            tool_overload_counter += 1
            if tool_overload_counter > 5:
                # print("new num_tool_overload", index)
                num_tool_overload += 1
                tool_overload_counter = 0
        else:
            tool_overload_counter = 0

        if error < mean - relax*std:
            tool_disengagement_counter += 1
            if tool_disengagement_counter > 5:
                num_tool_disengagement += 1
                tool_disengagement_counter = 0
        else:
            tool_disengagement_counter = 0
        
        if prev_base_load!= None and prev_current_load != None:
            dy = abs(prev_base_load - x)
            dx = abs(prev_current_load - y)
            if dx == 0:
                continue
            slope = dy/dx
            if slope > 2:
                num_roc += 1
                if num_roc >=2 :
                    num_roc = 0
            else:
                num_roc = 0
                
            
        prev_base_load = x
        prev_current_load = y

    data = [
           {'risk_type': 'tool_disengagement', 'risk_count': num_tool_disengagement, 'method': 'New Method'},
           {'risk_type': 'tool_overload', 'risk_count': num_tool_overload, 'method': 'New Method'},
           {'risk_type': 'roc', 'risk_count': num_roc, 'method': 'New Method'},
    ]
    df = pd.DataFrame.from_dict(data)
    return df

In [100]:
def mark_risk_old(current_load, new_base, t):

    tool_disengagement_counter = 0
    tool_overload_counter = 0
    roc_counter = 0

    num_tool_disengagement = 0
    num_tool_overload = 0
    num_roc = 0

    prev_base_load = None
    prev_current_load = None

    for index, (x, y) in enumerate(zip(current_load[:,t,0], new_base[:, t, 0])):
        # error = x - y
        # print(error)
        if x > 1.25*y:
            tool_overload_counter += 1
            if tool_overload_counter > 5:
                # print("num_tool_overload", index, x, y, 1.25*x, .75*x)
                num_tool_overload += 1
                tool_overload_counter = 0
        else:
            tool_overload_counter = 0

        if x < .75*y:
            tool_disengagement_counter += 1
            if tool_disengagement_counter > 5:
                # print("num_tool_diseng", index, x, y, 1.25*x, .75*x)
                num_tool_disengagement += 1
                tool_disengagement_counter = 0
        else:
            tool_disengagement_counter = 0
        
        if prev_base_load!= None and prev_current_load != None:
            dy = abs(prev_base_load - x)
            dx = abs(prev_current_load - y)
            if dx == 0:
                continue
            slope = dy/dx
            
            # print(slope, dy, dx)
            if slope > 2:
                num_roc += 1
                if num_roc >=2 :
                    num_roc = 0
            else:
                num_roc = 0
                
            
        prev_base_load = x
        prev_current_load = y

    # risk = {
    #     'tool_disengagement' : num_tool_disengagement,
    #     'tool_overload': num_tool_overload,
    #     'roc': num_roc,
    #     'method': 'Old Method'
    # }
    # pd.DataFrame.from_dict(risk)
    # return risk
    data = [
           {'risk_type': 'tool_disengagement', 'risk_count': num_tool_disengagement, 'method': 'Old Method'},
           {'risk_type': 'tool_overload', 'risk_count': num_tool_overload, 'method': 'Old Method'},
           {'risk_type': 'roc', 'risk_count': num_roc, 'method': 'Old Method'},
    ]
    df = pd.DataFrame.from_dict(data)
    return df

In [101]:
# number of risks from current methodology on production

In [102]:
display_seq(df_risk_base_scaled_x, df_risk_test_scaled_x, 4)

Number of anomalies:  431


In [103]:
# risks from new methodology, here red (predicted) line is the basline generated by the model while green line is fixed baseline from current method

In [104]:
df_risk_test_scaled_x_predicted = model.predict(df_risk_test_scaled_x)
display_prediction_new(df_risk_test_scaled_x, df_risk_test_scaled_x_predicted, df_risk_base_scaled_x, 4, mean, std, 2)

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
Number of anomalies:  6


In [105]:
def classify_anomaly(current_load, new_base, old_base, t, mean, std, relax):
    new_anomalies = mark_risk_new(current_load, new_base, t, mean, std, relax)
    # print(new_anomalies)
    old_anomalies = mark_risk_old(current_load, old_base, t)
    concatenated_df = pd.concat([old_anomalies, new_anomalies])
    
    # Reset the index of the concatenated DataFrame
    concatenated_df.reset_index(drop=True, inplace=True)
    fig = px.bar(concatenated_df, x="method", y="risk_count", color="risk_type", title="risks")
    fig.show()
    
    # print(new_anomalies)
    # print(old_anomalies)

In [106]:
classify_anomaly(df_risk_test_scaled_x, df_risk_test_scaled_x_predicted, df_risk_base_scaled_x, 4, mean, std, 2)

In [107]:
display_prediction(noise_data, predicted_load_test_noise, 4,  mean, std, 2)

Number of anomalies:  161


In [108]:
def get_cr_run_current_load_and_baseline(base_cr, current_cr, dfN10_load_filtered_cr):
    basecr = dfN10_load_filtered_cr[dfN10_load_filtered_cr['cr_numeric'].isin([base_cr])]
    current_cr = dfN10_load_filtered_cr[dfN10_load_filtered_cr['cr_numeric'].isin([current_cr])]

    basecr = basecr[['load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution',]]
    current_cr = current_cr[['load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution',]]
    
    scaler = MinMaxScaler()
    data_scaled = scaler.fit(basecr)
    # Scaling dataset according to weights of train data
    basecr_scaled = data_scaled.transform(basecr)
    current_cr_scaled = data_scaled.transform(current_cr)
    return basecr_scaled, current_cr_scaled

In [109]:
def plot_new_and_old_risks(base_cr, current_cr, dfN10_load_filtered_cr, relax):
    base, curr = get_cr_run_current_load_and_baseline(base_cr, current_cr, dfN10_load_filtered_cr)
    base_x, base_y = to_sequences(base, 10)
    curr_x, curr_y = to_sequences(curr, 10)
    predicted = model.predict(curr_x)

    display_seq(base_x, curr_x, 4)
    display_prediction_new(curr_x, predicted, base_x, 4, mean, std, relax)
    classify_anomaly(curr_x, predicted, base_x, 4, mean, std, relax)

In [110]:
plot_new_and_old_risks(3, 12, dfN10_load, 2)

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
Number of anomalies:  305


Number of anomalies:  13


In [111]:
plot_new_and_old_risks(3, 13, dfN10_load, 2)

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Number of anomalies:  476


Number of anomalies:  5


In [112]:
plot_new_and_old_risks(2, 14, dfN10_load, 2)

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
Number of anomalies:  445


Number of anomalies:  4


In [113]:
plot_new_and_old_risks(3, 15, dfN10_load, 2)

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
Number of anomalies:  431


Number of anomalies:  8


In [114]:
plot_new_and_old_risks(3, 16, dfN10_load, 2)

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
Number of anomalies:  433


Number of anomalies:  13


In [115]:
plot_new_and_old_risks(3, 16, dfN10_load, 2)

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
Number of anomalies:  433


Number of anomalies:  13


In [116]:
#dilation
def plot_new_and_old_risks_for_dilation(base_cr, current_cr, dfN10_load_filtered_cr, prob, dup_count, relax):
    base, curr = get_cr_run_current_load_and_baseline(base_cr, current_cr, dfN10_load_filtered_cr)
    curr = duplicate_rows_with_probability(curr, prob, dup_count)
    base_x, base_y = to_sequences(base, 10)
    curr_x, curr_y = to_sequences(curr, 10)
    predicted = model.predict(curr_x)

    display_seq(base_x, curr_x, 4)
    display_prediction_new(curr_x, predicted, base_x, 4, mean, std, relax)
    classify_anomaly(curr_x, predicted, base_x, 4, mean, std, relax)

In [117]:
plot_new_and_old_risks_for_dilation(3, 12, dfN10_load, 0.1, 5, 2)

 5/48 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step

48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step
Number of anomalies:  560


Number of anomalies:  8


In [123]:
def get_data(machines, programs, dates):
    # machines= ['SL40309_015']
    # programs= ['O0005(5303-005-C)']
    # nsequences= ['N1']
    # dates= ['CR1_to_CR4', 'CR5_to_CR8', 'CR9_to_CR12', 'CR12_to_CR16']
    base_name = 'data'
    paths = []
    
    # data_SL40309_015_O0005(5303-005-C)_CR12_to_CR16
    
    for machine in machines:
        for program in programs:
                for date in dates:
                    path = f'{base_name}_{machine}_{program}_{date}.csv'
                    paths.append(path)
    print(paths)
    
    dfs = []
    
    for path in paths:
        data = pd.read_csv(f'../data/{path}')
        dfs.append(data)
        # print(len(data))
        
    data = pd.concat(dfs, ignore_index=True)
    
    data = data.pivot(index=['custom_id', 'timestamp', 'status', 'cr'], columns=['name', 'workstationcomponent'], values='value')
    data = data.reset_index()
    data.columns = [f'{col}_{comp}' if comp != '' else col for col, comp in data.columns]

    split_columns = data['status'].str.split('/', expand=True)
    data[['program_name', 'nsequence', 'execution']] = split_columns[[0,1,2]]
    
    data['load_Rotary_C5'] = data['load_Rotary_C5'].astype(float)
    data['position_Linear_Z'] = data['position_Linear_Z'].astype(float)
    data['position_Linear_X'] = data['position_Linear_X'].astype(float)
    data['spindlespeed_actual_Rotary_C5'] = data['spindlespeed_actual_Rotary_C5'].astype(float)
    data['pathfeedrate_Path_Path_1'] = data['pathfeedrate_Path_Path_1'].astype(float)
    
    data = data.ffill()
    return data

In [119]:
# machines_diff = ['NL300005_007']
# programs_diff = ['O4073(3980-050-C)']
# dates_diff = ['CR1_to_CR4'] #, 'CR5_to_CR8', 'CR9_to_CR12', 'CR12_to_CR16']
# # data_NL250007_013_O4125(5211-020-D)_N1_cr_1_to_5
# # data_NL300005_007_O4073(3980-050-C)_CR1_to_CR4

# data = get_data(machines_diff, programs_diff, dates_diff)

In [124]:
# raw_data_SL40309_015_O0020(5304-020-F)_N10_2024-03-19

machines_diff = ['SL40309_015']
programs_diff = ['O0020(5304-020-F)']
dates_diff = ['N10_2024-03-19'] #, 'CR5_to_CR8', 'CR9_to_CR12', 'CR12_to_CR16']
# data_NL250007_013_O4125(5211-020-D)_N1_cr_1_to_5
# data_NL300005_007_O4073(3980-050-C)_CR1_to_CR4

data = get_data(machines_diff, programs_diff, dates_diff)

['data_SL40309_015_O0020(5304-020-F)_N10_2024-03-19.csv']


In [125]:
len(data)

25024

In [126]:
def check_model_on_different_program(machines, programs, dates):
    data = get_data(machines_diff, programs_diff, dates_diff)
    data = data[(data['execution'] == 'ACTIVE') & 
            (data['spindlespeed_actual_Rotary_C5'] != 0) & 
            (data['pathfeedrate_Path_Path_1'] != 0) & 
            (data['load_Rotary_C5'] > 0) & 
            (data['pathfeedrate_Path_Path_1'] <= 1000)]
    
    # data_c = data[data['program_name'].isin(['O4125(5211-020-D)'])]
    # dfN10 = data_c[data_c['nsequence'].isin(['N1'])]
    data.dropna(subset=['load_Rotary_C5'],inplace=True)
    
    df_diff = data[['timestamp', 'load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution', 'program_name', 'nsequence', 'cr']]
    df_diff = df_diff.drop_duplicates(subset=['timestamp'], keep ='first')
    df_diff.reset_index(drop=True)

    df_diff['index'] = df_diff.groupby(['cr']).cumcount()
    
    fig = px.line(df_diff, x="index", y="load_Rotary_C5", color='cr')
    fig.update_layout(title_text="Load Vs Index")
    fig.show()

    df_diff['execution'] = LE.fit_transform(df_diff['execution'])
    df_diff.fillna(0, inplace=True)

    df_diff[['load_Rotary_C5']]= scaler1.fit_transform(df_diff[['load_Rotary_C5']])

    df_diff = df_diff[['load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution',]]

    # scalar = MinMaxScaler()
    # data_scaled = scaler.fit(df_diff)
    # df_diff_scaled = data_scaled.transform(df_diff)
    df_diff_scaled = data_scaled.transform(df_diff)
    print(f"Max Before {df_diff.max()}")
    print(f"Max After {df_diff_scaled.max()}")
    # df_diff_scaled[-3]
    df10_trainX_diff, df10_trainY_diff = to_sequences(df_diff_scaled, 10)
    predicted_diff = model.predict(df10_trainX_diff)
    # print(predicted_diff[0:10])
    display_prediction(df10_trainX_diff, predicted_diff, 4, mean, std, 2)
    


In [127]:
machines_diff = ['SL40309_015']
programs_diff = ['O0020(5304-020-F)']
dates_diff = ['N10_2024-03-19'] #, 'CR5_to_CR8', 'CR9_to_CR12', 'CR12_to_CR16']
check_model_on_different_program(machines_diff, programs_diff, dates_diff)

['data_SL40309_015_O0020(5304-020-F)_N10_2024-03-19.csv']


Max Before load_Rotary_C5                     1.0000
position_Linear_X                 24.8753
position_Linear_Z                 17.7050
spindlespeed_actual_Rotary_C5     57.0000
pathfeedrate_Path_Path_1         259.5800
execution                          0.0000
dtype: float64
Max After 1.3529411764705879
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
Number of anomalies:  28


In [128]:
machines_diff = ['NL250007_013']
programs_diff = ['O4125(5211-020-D)']
# data_SL40309_015_O0005(5303-005-C)_CR1_to_CR4
# data_NL250007_013_O4125(5211-020-D)_N1_cr_1_to_5
dates_diff = ['N1_cr_1_to_5']
check_model_on_different_program(machines_diff, programs_diff, dates_diff)

['data_NL250007_013_O4125(5211-020-D)_N1_cr_1_to_5.csv']


Max Before load_Rotary_C5                     1.00000
position_Linear_X                453.49922
position_Linear_Z                320.95440
spindlespeed_actual_Rotary_C5     73.00000
pathfeedrate_Path_Path_1         629.00000
execution                          0.00000
dtype: float64
Max After 22.652655866078028
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
Number of anomalies:  2581


In [129]:
machines_diff = ['SL40309_015']
programs_diff = ['O0020(5304-020-F)']
dates_diff = ['N10_2024-03-19'] #, 'CR5_to_CR8', 'CR9_to_CR12', 'CR12_to_CR16']

# machines_diff = ['SL40309_015']
# programs_diff = ['O0005(5303-005-C)']
# # data_SL40309_015_O0005(5303-005-C)_CR1_to_CR4
# dates_diff = ['CR1_to_CR4', 'CR5_to_CR8', 'CR9_to_CR12', 'CR12_to_CR16']

data = get_data(machines_diff, programs_diff, dates_diff)
data = data[(data['execution'] == 'ACTIVE') & 
        (data['spindlespeed_actual_Rotary_C5'] != 0) & 
        (data['pathfeedrate_Path_Path_1'] != 0) & 
        (data['load_Rotary_C5'] > 0) & 
        (data['pathfeedrate_Path_Path_1'] <= 1000)]

# data_c = data[data['program_name'].isin(['O4125(5211-020-D)'])]
# dfN10 = data_c[data_c['nsequence'].isin(['N1'])]
data.dropna(subset=['load_Rotary_C5'],inplace=True)

df_diff = data[['timestamp', 'load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution', 'program_name', 'nsequence', 'cr']]
df_diff = df_diff.drop_duplicates(subset=['timestamp'], keep ='first')
df_diff.reset_index(drop=True)

df_diff['index'] = df_diff.groupby(['cr']).cumcount()

df_diff['execution'] = LE.fit_transform(df_diff['execution'])
df_diff.fillna(0, inplace=True)
# scaler1 = MinMaxScaler()

df_diff[['load_Rotary_C5']]= scaler1.fit_transform(df_diff[['load_Rotary_C5']])
df_diff = df_diff[['load_Rotary_C5', 'position_Linear_X', 'position_Linear_Z', 'spindlespeed_actual_Rotary_C5', 'pathfeedrate_Path_Path_1', 'execution',]]

# scalar = MinMaxScaler()
# data_scaled = scaler.fit(df_diff)
df_diff_scaled = data_scaled.transform(df_diff)


['data_SL40309_015_O0020(5304-020-F)_N10_2024-03-19.csv']


In [130]:
df_diff_scaled.max()

np.float64(1.3529411764705879)

In [131]:
df_diff_scaled

array([[ 4.50980392e-01,  1.02231367e+00,  9.56811298e-01,
         7.65550239e-02,  1.06574739e+00,  0.00000000e+00],
       [ 6.44257703e-02,  7.57710901e-01,  7.41269431e-01,
         7.65550239e-02,  1.06574739e+00,  0.00000000e+00],
       [ 6.44257703e-02,  7.57710901e-01,  4.87006094e-01,
         7.89473684e-02,  1.06566433e+00,  0.00000000e+00],
       ...,
       [ 6.44257703e-02,  7.20150223e-01,  2.28652576e-01,
         5.98086124e-02, -8.30668273e-05,  0.00000000e+00],
       [ 3.22128852e-02,  7.19675783e-01,  2.69279792e-01,
         5.02392344e-02,  7.80329775e-01,  0.00000000e+00],
       [ 3.22128852e-02,  7.61686210e-01,  5.18379174e-01,
         4.78468900e-02,  9.83428168e-01,  0.00000000e+00]],
      shape=(3482, 6))

In [132]:
df_diff

,load_Rotary_C5,position_Linear_X,position_Linear_Z,spindlespeed_actual_Rotary_C5,pathfeedrate_Path_Path_1,execution
18,0.333333,20.3819,17.7050,45.0,256.83,0
23,0.047619,15.0836,12.4458,45.0,256.83,0
29,0.047619,15.0836,6.2418,46.0,256.81,0
88,0.047619,11.4611,2.0000,46.0,236.20,0
93,0.047619,11.4611,-0.0620,46.0,236.20,0
...,...,...,...,...,...,...
24773,0.047619,14.3431,-0.0620,38.0,0.21,0
24775,0.047619,14.3315,-0.0620,38.0,0.21,0
24776,0.047619,14.3315,-0.0620,38.0,0.21,0
24987,0.023810,14.3220,0.9293,34.0,188.11,0


In [133]:
df_diff.max()

load_Rotary_C5                     1.0000
position_Linear_X                 24.8753
position_Linear_Z                 17.7050
spindlespeed_actual_Rotary_C5     57.0000
pathfeedrate_Path_Path_1         259.5800
execution                          0.0000
dtype: float64

In [135]:
df10_trainX_diff, df10_trainY_diff = to_sequences(df_diff_scaled, 10)
predicted_diff = model.predict(df10_trainX_diff)
display_prediction(df10_trainX_diff, predicted_diff, 4, mean, std, 2)

109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
Number of anomalies:  28
